# Convert a list of fits files into a WWT ingestible WTML file

We will use the `toasty.tile_fits` utility function reproject the data into the TOAST Tiles. It can also create a HiPS Tiles

Place your files in a directory called `source`

The output directory is `gal_plane_hips'


We have to do a couple extra steps to avoid a couple bugs
- Even though the tiler does not care what the projection is (it get's reprojected), the WTML builder looks at the original header and will break if it sees something that is not a TAN projected. Since these are tiny, SIN = TAN. But if near a pole, this assumption will break
- We need to set the blank value. by default it puts nan in empty pixels, but the WWT shader does not reliably capture nan. 

_Both of these issues should get fixed upstream to avoid this extra processing_

**You should be able to just run the cells in order to have this complete**

In [ ]:
from pathlib import Path
from toasty import tile_fits, TilingMethod
from astropy.io import fits
import numpy as np


files = list(Path('./source').glob('*.fits'))
print(f"There are {len(files)} fits files")


In [ ]:
# This is a cheat. tile_fits works
# with whatever coordinate system. But the
# WTML builder does not know this and errors when it sees a SIN in the header
relabeled = []
for p in files:
    out = Path(p).with_suffix(".tanlabel.fits")
    if not out.exists():
        with fits.open(p) as hdul:
            hdul[0].header["CTYPE1"] = "RA---TAN"
            hdul[0].header["CTYPE2"] = "DEC--TAN"
            hdul.writeto(out, overwrite=True)
    relabeled.append(str(out))

In [ ]:
# https://toasty.readthedocs.io/en/latest/api/toasty.tile_fits.html#toasty.tile_fits
out_dir, builder = tile_fits(
    relabeled,
    out_dir="gal_plane_toast",
    tiling_method=TilingMethod.TOAST,   # force TOAST. Can also do TilingMethod.HIPS which requires a java JRE
    override=True,  # overwrite existing files
    # blankval = 0 default None so nan will be used
)
print(f"Done at level {builder.imgset.tile_levels}; wrote {out_dir}/index_rel.wtml")
# wwtdatatool wtml rewrite-urls [-h] INPUT-WTML BASE-URL OUTPUT-WTML

In [ ]:
# eventually this should not be necessary, but currently WWT igonres nan and paints them as 0 values
data_dir = Path('gal_plane_toast')
BLANK_VALUE = -9999.0

for path in data_dir.rglob("*.fits"):
    with fits.open(path, mode="update") as hdul:
        hdu = hdul[0]
        data = hdu.data
        mask = np.isnan(data)
        if not mask.any():
            continue
        data[mask] = BLANK_VALUE
        hdu.data = data
        hdu.header["BLANK"] = BLANK_VALUE
        hdul.flush()
    # print(path)


In [ ]:
# clean up the tanlabel files
import shutil

for p in relabeled:
    Path(p).unlink()